# download source files to volume

In [0]:
from datetime import datetime, timezone
import urllib.request
import tempfile
import os
import json
import uuid

from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
%run ./utils/functions

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "pb_threat_intelligence")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

In [0]:
feed_links = {
    'feed_a': 'https://raw.githubusercontent.com/StevenBlack/hosts/master/hosts',
    'feed_b': 'https://malware-filter.gitlab.io/malware-filter/urlhaus-filter-agh.txt',
    'feed_c': 'https://badmojr.gitlab.io/1hosts/Lite/adblock.txt',
    'feed_d': 'https://raw.githubusercontent.com/jdlingyu/ad-wars/master/hosts',
}

In [0]:
# one identifier and timestamp shared by all feeds in this run.
ingestion_ts = datetime.now(timezone.utc)

run_id = (
    f"{ingestion_ts:%H%M%SZ}_"
    f"{uuid.uuid4().hex[:8]}"
)

volume_root = Path(
    f"/Volumes/{catalog}/{schema}/feed_snapshots"
)

snapshot_dir = volume_root / f"{ingestion_ts:%Y%m%d}" / run_id
snapshot_dir.mkdir(parents=True, exist_ok=False)

errors = {}

# attempt every feed, even if an earlier feed download failed.
for name, url in feed_links.items():
    try:
        path = download_feed(
            name=name,
            url=url,
            snapshot_dir=snapshot_dir,
            run_id=run_id,
        )

    except RuntimeError as e:
        errors[name] = str(e)

print(f"Run ID: {run_id}")
print(f"Snapshot directory: {snapshot_dir}")

if errors:
    # stop before downstream processing
    raise RuntimeError(
        f"Failed feeds: {', '.join(errors)}. "
        f"Check metadata in {snapshot_dir}"
    )
else:
    print("All feeds downloaded successfully")

Downloaded feed_a -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/20261007/203832Z_0ccefab2/feed_a.txt
Downloaded feed_b -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/20261007/203832Z_0ccefab2/feed_b.txt
Downloaded feed_c -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/20261007/203832Z_0ccefab2/feed_c.txt
Downloaded feed_d -> /Volumes/workspace/pb_threat_intelligence/feed_snapshots/20261007/203832Z_0ccefab2/feed_d.txt
Run ID: 203832Z_0ccefab2
Snapshot directory: /Volumes/workspace/pb_threat_intelligence/feed_snapshots/20261007/203832Z_0ccefab2
All feeds downloaded successfully
